# Data Organize — clean & feature-prep

Read the raw race table from `data_loading` (`data/raw/races.parquet`), clean it, add a few leakage-safe features, and save `data/processed/features.parquet` for `model_training`.

**Leakage rule:** a feature for a race may only use information from *earlier* races. That's why every form feature does `groupby(...).shift(1)` before rolling.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from f1pred import paths

races = pd.read_parquet(paths.RAW / "races.parquet")
races = races.sort_values(["season", "round"]).reset_index(drop=True)

assert len(races) > 0 and {"season", "round", "DriverId"} <= set(races.columns)
races.head()

## 1. Clean

Fix the known gotchas and define the prediction target (finishing order).

In [ ]:
df = races.copy()

# grid 0 == pit-lane start -> treat as back of that race's grid, not pole
df["grid_start"] = df["grid_position"].replace(0, np.nan)
df["grid_start"] = df["grid_start"].fillna(
    df.groupby(["season", "round"])["grid_start"].transform("max") + 1
)

# did the driver get a classified finish? (ClassifiedPosition is numeric only if so)
df["finished"] = df["ClassifiedPosition"].str.fullmatch(r"\d+").fillna(False)

# best qualifying lap as seconds (model-friendly number)
df["quali_seconds"] = df["best_quali"].dt.total_seconds()

# target = finishing order; DNFs ranked at the back of their race
df["n_drivers"] = df.groupby(["season", "round"])["DriverId"].transform("size")
df["target"] = df["race_position"].fillna(df["n_drivers"])

assert df["grid_start"].notna().all() and df["target"].notna().all()

## 2. Features

A handful of obvious, leakage-safe signals. Form features look only at past races (`shift(1)` first), so a driver's/team's debut race is `NaN` — which is correct.

In [ ]:
# driver recent form: mean finish over the previous 5 races (never the current one)
df = df.sort_values(["DriverId", "season", "round"])
df["driver_form"] = df.groupby("DriverId")["target"].transform(
    lambda s: s.shift(1).rolling(5, min_periods=1).mean()
)

# team recent form: same idea, grouped by team
df = df.sort_values(["TeamId", "season", "round"])
df["team_form"] = df.groupby("TeamId")["target"].transform(
    lambda s: s.shift(1).rolling(5, min_periods=1).mean()
)

df = df.sort_values(["season", "round"]).reset_index(drop=True)

# qualifying gap to pole (seconds), within each race
df["quali_gap"] = df["quali_seconds"] - df.groupby(["season", "round"])["quali_seconds"].transform("min")

# grid penalty: positions lost between qualifying and the actual start
df["grid_penalty"] = df["grid_start"] - df["quali_position"]

assert {"driver_form", "team_form", "quali_gap", "grid_penalty"} <= set(df.columns)
assert df["driver_form"].isna().any()   # debut races carry no prior form (leakage-safe)

## 3. Save

Write the model-ready table for `model_training`.

In [ ]:
out_dir = paths.DATA / "processed"
out_dir.mkdir(parents=True, exist_ok=True)
out = out_dir / "features.parquet"
df.to_parquet(out, index=False)

assert out.exists()